In [ ]:
import pandas as pd
import datetime
s = pd.Series([
    "2025-03-14", "14/03/2025", "03/04/2025", "2025-11-02",
    "31/12/2024", None, "N/A", "2025-13-45", "07/07/2025",
    "1/2/2025", "2025-06-30", "  22/08/2025  ",
])


N/A
2025-13-45


C:\Users\loren\AppData\Local\Temp\ipykernel_38320\3859584512.py:10: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  new_list.append(pd.to_datetime(val))
C:\Users\loren\AppData\Local\Temp\ipykernel_38320\3859584512.py:10: UserWarning: Parsing dates in   %d/%m/%Y   format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  new_list.append(pd.to_datetime(val))


([Timestamp('2025-03-14 00:00:00'),
  Timestamp('2025-03-14 00:00:00'),
  Timestamp('2025-03-04 00:00:00'),
  Timestamp('2025-11-02 00:00:00'),
  Timestamp('2024-12-31 00:00:00'),
  Timestamp('2025-07-07 00:00:00'),
  Timestamp('2025-01-02 00:00:00'),
  Timestamp('2025-06-30 00:00:00'),
  Timestamp('2025-08-22 00:00:00')],
 {'total': 12, 'Nan_value': 1, 'Not_parse': 2})

In [ ]:
def parse_date(s):
    result ={}
    new_list = []
    result['total']= len(s)
    result['Nan_value']= len(s[s.isna()])
    result['Not_parse']=0
    s_with=s[s.isna()==False]
    for val in s_with:
        try:
            new_list.append(pd.to_datetime(val))
        except   Exception as e:
            print(val)
            result['Not_parse'] += 1
    return new_list,result

parse_date(s)

In [ ]:
import re
import pandas as pd

_SLASH = re.compile(r"^\s*(\d{1,4})[/-](\d{1,2})[/-](\d{2,4})\s*$")

def parse_dates(s):
    rep = {"total": len(s), "null_input": int(s.isna().sum())}

    # 1. rileva la convenzione sull'INTERA colonna, non riga per riga
    firsts, seconds = [], []
    for v in s.dropna().astype(str):
        m = _SLASH.match(v)
        if m and len(m.group(1)) <= 2:
            firsts.append(int(m.group(1)))
            seconds.append(int(m.group(2)))

    dayfirst = any(f > 12 for f in firsts)
    rep["dayfirst_detected"] = dayfirst

    # 2. ambigue = entrambi i componenti <= 12
    rep["ambiguous"] = sum(1 for f, sec in zip(firsts, seconds) if f <= 12 and sec <= 12)

    # 3. parse UNICO su tutta la Series, indice preservato
    out = pd.to_datetime(s, dayfirst=dayfirst, format="mixed", errors="coerce")

    rep["parsed"] = int(out.notna().sum())
    rep["unparseable"] = rep["total"] - rep["parsed"] - rep["null_input"]
    return out, rep

In [54]:
claims = pd.DataFrame({
    "claim_id":   ["c1","c2","c3","c4","c5","c6","c7","c8"],
    "policy_id":  ["p1","p2","p1","p9","p3","p2","p4","p1"],
    "reported_at":pd.to_datetime(["2025-01-15","2025-02-20","2025-04-02",
                                   "2025-05-11","2025-07-30","2025-08-01",
                                   "2025-11-19",None]),
    "amount":     [1240.50, 980.0, None, 4500.0, 220.0, 1500.0, 75.0, 300.0],
    "status":     ["closed","closed","closed","closed","open","closed","closed","closed"],
})

policies = pd.DataFrame({
    "policy_id":  ["p1","p2","p3","p4"],
    "customer_id":["u1","u2","u1","u3"],
    "region":     ["north","north","south","south"],
})

In [52]:
def check_null(df: pd.DataFrame, con_name:str):
    return df[df[con_name].isna()==False],len(df[df[con_name].isna()])

In [55]:
check_null(claims,"claim_id")

(  claim_id policy_id reported_at  amount  status
 0       c1        p1  2025-01-15  1240.5  closed
 1       c2        p2  2025-02-20   980.0  closed
 2       c3        p1  2025-04-02     NaN  closed
 3       c4        p9  2025-05-11  4500.0  closed
 4       c5        p3  2025-07-30   220.0    open
 5       c6        p2  2025-08-01  1500.0  closed
 6       c7        p4  2025-11-19    75.0  closed
 7       c8        p1         NaT   300.0  closed,
 0)

In [91]:
def report(claims, policies):
    stats={}
    for col in claims.columns:
        claims,report= check_null(claims,col)
        stats[f"null_{col}"]= report
    claims.status=claims.status.apply(lambda x: x.lower())
    stats['not_closed']=len(claims[claims.status != "closed"])
    claims = claims[claims.status == "closed"]
    merge_left=claims.merge(policies,"left")
    ml= merge_left[merge_left.customer_id.isna()==False]
    ml_na = merge_left[merge_left.customer_id.isna()]
    stats['amount_orphan']=ml_na['amount'].sum()
    stats['number_orphan']=len(ml_na)
    ml['quarter']=ml.reported_at.dt.quarter
    total_closed_amount=ml.groupby(by=['region','quarter'])['amount'].sum()
    distinct_customers=ml.groupby(by=['region','quarter'])['customer_id'].nunique()

    return total_closed_amount,distinct_customers,stats


In [92]:
vals = pd.Series([
    "1,240.50",   # US: 1240.50
    "1.240,50",   # IT: 1240.50
    "  980 ",     # 980
    "1,240",      # ambiguo-ma-risolvibile: 1240
    "0,75",       # 0.75
    "12.345.678,90",
    "N/A", None, "-", "",
    "(1.500,00)", # negativo tra parentesi
    "3 200,00",   # separatore spazio
])

In [ ]:
df="1,240.50  "

In [114]:
df.isnumeric()

True

In [111]:
arr= df.strip().replace(' ','').replace('.',',').split(',')

In [137]:
def parse_amounts(vals):
    result=[]
    for df in vals:
        arr= str(df).strip().replace(' ','').replace(',','.').split('.')
        res=''
        try:
            for a in range(len(arr)):
                if str(arr[a]).isnumeric() == False:
                    print("property")
                    raise Exception()
                if a == len(arr)-1 and len(arr) != 1:
                    res = f"{res}.{arr[a]}"
                else:
                    res = f"{res}{arr[a]}"
            print(res)
            result.append(float(res))
        except Exception as e:
            continue
    return result

In [138]:
parse_amounts(vals)

1240.50
1240.50
980
1.240
0.75
12345678.90
property
property
property
property
property
3200.00


[1240.5, 1240.5, 980.0, 1.24, 0.75, 12345678.9, 3200.0]

In [139]:
float('-150')

-150.0